# Data-to-Model Comparison

Este notebook **reusa as mesmas funções** de:
- **aprendizado (fit)**: `K_rbf`, `nll_gp_rbf`, `fit_gp_rbf_hyperparams`
- **predição**: `gp_predict_rbf`
- **plot do fit (ROOT/JSROOT)**: `plot_gp_fit_root_inline`

Dados experimentais:

`HEPData-ins2907010-v1-Figure_4a_cent7.csv` 

Saídas:
- Plot ROOT: dados + GP mean ± 2σ  
- Plot ROOT: **σ_exp(pT)** vs **σ_GP,obs(pT)**  
- Tabela: comparação ponto-a-ponto (experimental vs GP)


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path


## 1) Dados HEPData (CSV)

In [2]:
DATA_FILE = Path("HEPData-ins2907010-v1-Figure_4a_cent7.csv")
if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {DATA_FILE.resolve()}\n"
        "Coloque o CSV no mesmo diretório deste notebook."
    )

df = pd.read_csv(
    DATA_FILE,
    comment="#",
    header=None,
    names=["pT","v0","stat_plus","stat_minus","sys_plus","sys_minus"],
)

df = df.astype(float).sort_values("pT").reset_index(drop=True)
df.head(10)


,pT,v0,stat_plus,stat_minus,sys_plus,sys_minus
0,0.55231,-0.044431,0.000092,-0.000092,0.001367,-0.001367
1,0.64791,-0.035065,0.000129,-0.000129,0.001148,-0.001148
2,0.74831,-0.023929,0.000124,-0.000124,0.000972,-0.000972
3,0.84819,-0.013116,0.000134,-0.000134,0.000803,-0.000803
4,0.94830,-0.002188,0.000137,-0.000137,0.000721,-0.000721
5,1.09310,0.012981,0.000127,-0.000127,0.000803,-0.000803
6,1.29330,0.033574,0.000182,-0.000182,0.001127,-0.001127
7,1.49340,0.052630,0.000166,-0.000166,0.001659,-0.001659
8,1.69350,0.070958,0.000203,-0.000203,0.001965,-0.001965
9,1.89360,0.088492,0.000279,-0.000279,0.002584,-0.002584


## 2) Barras de erro experimentais

In [3]:
def symmetrize_err(plus, minus):
    plus = np.asarray(plus, dtype=float)
    minus = np.asarray(minus, dtype=float)
    return 0.5 * (np.abs(plus) + np.abs(minus))

df["sigma_stat"] = symmetrize_err(df["stat_plus"], df["stat_minus"])
df["sigma_sys"]  = symmetrize_err(df["sys_plus"], df["sys_minus"])
df["sigma_exp"]  = np.sqrt(df["sigma_stat"]**2 + df["sigma_sys"]**2)

x = df["pT"].to_numpy(dtype=np.float64)
y = df["v0"].to_numpy(dtype=np.float64)

df[["pT","v0","sigma_stat","sigma_sys","sigma_exp"]].head(10)


,pT,v0,sigma_stat,sigma_sys,sigma_exp
0,0.55231,-0.044431,0.000092,0.001367,0.001370
1,0.64791,-0.035065,0.000129,0.001148,0.001155
2,0.74831,-0.023929,0.000124,0.000972,0.000980
3,0.84819,-0.013116,0.000134,0.000803,0.000814
4,0.94830,-0.002188,0.000137,0.000721,0.000733
5,1.09310,0.012981,0.000127,0.000803,0.000813
6,1.29330,0.033574,0.000182,0.001127,0.001142
7,1.49340,0.052630,0.000166,0.001659,0.001667
8,1.69350,0.070958,0.000203,0.001965,0.001975
9,1.89360,0.088492,0.000279,0.002584,0.002599


## 3) GP (fit) 

In [4]:
import numpy as np
import scipy
from scipy.linalg import cho_factor, cho_solve
from scipy.optimize import minimize

# -----------------------------
# RBF / SE kernel matrix
# -----------------------------
def K_rbf(x, sigma_f, ell):
    x = np.asarray(x, dtype=np.float64).reshape(-1, 1)
    sqdist = (x - x.T) ** 2
    return (sigma_f**2) * np.exp(-0.5 * sqdist / (ell**2))

# -----------------------------
# Log marginal likelihood (and negative)
# Params are in log-space for stability:
# theta = [log_sigma_f, log_ell, log_sigma_n]
# -----------------------------
def nll_gp_rbf(theta, x, y, jitter=1e-8):
    log_sigma_f, log_ell, log_sigma_n = theta
    sigma_f = np.exp(log_sigma_f)
    ell     = np.exp(log_ell)
    sigma_n = np.exp(log_sigma_n)

    y = np.asarray(y, dtype=np.float64).reshape(-1)
    n = len(y)

    K = K_rbf(x, sigma_f, ell)
    Ky = K + (sigma_n**2 + jitter) * np.eye(n)

    # (1) garante simetria numérica
    Ky = 0.5 * (Ky + Ky.T)

    # (2) se Cholesky falhar, devolve "péssimo" para o otimizador evitar
    try:
        c, lower = cho_factor(Ky, lower=True, check_finite=False)
        alpha = cho_solve((c, lower), y, check_finite=False)
        logdet = 2.0 * np.sum(np.log(np.diag(c)))
        lml = -0.5 * y.dot(alpha) - 0.5 * logdet - 0.5 * n * np.log(2*np.pi)
        return -lml
    except np.linalg.LinAlgError:
        return 1e50


# -----------------------------
# Fit helper
# -----------------------------
def fit_gp_rbf_hyperparams(x, y, *, init=None, bounds=None, jitter=1e-10):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)

    # Heuristic init if not provided
    if init is None:
        ystd = np.std(y) if np.std(y) > 0 else 1.0
        init_sigma_f = ystd
        init_ell     = 1.0
        init_sigma_n = 0.3 * ystd + 1e-6
        init = np.log([init_sigma_f, init_ell, init_sigma_n])

    # Reasonable bounds in log-space
    if bounds is None:
        # sigma_f in [1e-6, 1e3], ell in [1e-3, 1e3], sigma_n in [1e-8, 1e3]
        bounds = [(-14, 7), (-7, 7), (-18, 7)]

    obj = lambda th: nll_gp_rbf(th, x, y, jitter=jitter)

    res = minimize(
        obj,
        x0=np.asarray(init, dtype=np.float64),
        method="L-BFGS-B",
        bounds=bounds,
    )

    log_sigma_f, log_ell, log_sigma_n = res.x
    sigma_f = float(np.exp(log_sigma_f))
    ell     = float(np.exp(log_ell))
    sigma_n = float(np.exp(log_sigma_n))

    # Return also LML at optimum
    lml = -nll_gp_rbf(res.x, x, y, jitter=jitter)

    out = {
        "sigma_f": sigma_f,
        "ell": ell,
        "sigma_n": sigma_n,
        "log_marginal_likelihood": float(lml),
        "success": bool(res.success),
        "message": res.message,
        "nfev": res.nfev,
    }
    return out



In [5]:
fit_real = fit_gp_rbf_hyperparams(x, y)

print("=== GP fit: HEPData Figure_4a_cent7 ===")
for k,v in fit_real.items():
    print(f"{k}: {v}")


=== GP fit: HEPData Figure_4a_cent7 ===
sigma_f: 0.13499087397010287
ell: 2.5889202995373806
sigma_n: 0.001338606659135485
log_marginal_likelihood: 120.86145809608395
success: True
message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
nfev: 84


## 4) GP (predict) 

In [6]:
import sys
print(sys.executable)

def gp_predict_rbf(x_train, y_train, x_test, *, sigma_f, ell, sigma_n, jitter=1e-10):
    x_train = np.asarray(x_train, dtype=np.float64)
    y_train = np.asarray(y_train, dtype=np.float64).reshape(-1)
    x_test  = np.asarray(x_test, dtype=np.float64)

    # Kernel matrices
    K   = K_rbf(x_train, sigma_f, ell)
    Ky  = K + (sigma_n**2 + jitter) * np.eye(len(x_train))

    K_s = (sigma_f**2) * np.exp(
        -0.5 * (x_test.reshape(-1,1) - x_train.reshape(1,-1))**2 / ell**2
    )
    K_ss = K_rbf(x_test, sigma_f, ell)

    # Cholesky
    c, lower = cho_factor(Ky, lower=True, check_finite=False)
    alpha = cho_solve((c, lower), y_train, check_finite=False)

    # Posterior mean
    mu = K_s @ alpha

    # Posterior covariance
    v = cho_solve((c, lower), K_s.T, check_finite=False)
    cov = K_ss - K_s @ v

    var = np.clip(np.diag(cov), 0.0, np.inf)

    return mu, var

/Users/chanayo/Library/Mobile Documents/com~apple~CloudDocs/Repository/Academia/Course-Codex/cern-root-student-course/.venv/bin/python


In [10]:
# malha para curva
x_star = np.linspace(x.min(), x.max(), 400)

mu_star, var_star_latent = gp_predict_rbf(
    x_train=x,
    y_train=y,
    x_test=x_star,
    sigma_f=fit_real["sigma_f"],
    ell=fit_real["ell"],
    sigma_n=fit_real["sigma_n"],
)

# incerteza observacional do GP (comparável à barra experimental)
sigma_star_obs = np.sqrt(var_star_latent + fit_real["sigma_n"]**2)

# também nos pontos experimentais (pra tabela)
mu_x, var_x_latent = gp_predict_rbf(
    x_train=x,
    y_train=y,
    x_test=x,
    sigma_f=fit_real["sigma_f"],
    ell=fit_real["ell"],
    sigma_n=fit_real["sigma_n"],
)
sigma_x_obs = np.sqrt(var_x_latent + fit_real["sigma_n"]**2)

(mu_star[:3], sigma_star_obs[:3])


(array([-0.04453192, -0.04219966, -0.03986064]),
 array([0.00161343, 0.00158148, 0.00155364]))

## 5) Plot da predição 

In [11]:
import ROOT
import numpy as np
import uuid

def plot_gp_fit_root_inline(
    x_train, y_train,
    x_star, mu, var,
    *,
    title="GP posterior mean ± 2σ",
    canvas_name="c_gp",
    canvas_title="GP fit",
    width=1100,
    height=450,
    nsigma=2.0,
):
    # ROOT config: inline
    ROOT.gROOT.SetBatch(True)
    ROOT.gStyle.SetOptStat(0)

    # Avoid canvas name collisions
    uid = uuid.uuid4().hex[:6]
    cname = f"{canvas_name}_{uid}"

    # Ensure numpy float64 1D
    x_train = np.asarray(x_train, dtype=np.float64).ravel()
    y_train = np.asarray(y_train, dtype=np.float64).ravel()
    x_star  = np.asarray(x_star,  dtype=np.float64).ravel()
    mu      = np.asarray(mu,      dtype=np.float64).ravel()
    var     = np.asarray(var,     dtype=np.float64).ravel()

    std = np.sqrt(np.clip(var, 0.0, np.inf))

    c = ROOT.TCanvas(cname, canvas_title, width, height)

    # Keep references alive for JSROOT
    c._objs = []

    # --- Data points
    g_data = ROOT.TGraph(len(x_train), x_train, y_train)
    g_data.SetName(f"g_data_{uid}")
    g_data.SetMarkerStyle(20)
    g_data.SetMarkerSize(1.0)
    g_data.SetTitle(f"{title};x;y")
    c._objs.append(g_data)

    # --- Mean curve
    g_mu = ROOT.TGraph(len(x_star), x_star, mu)
    g_mu.SetName(f"g_mu_{uid}")
    g_mu.SetLineWidth(2)
    c._objs.append(g_mu)

    # --- Uncertainty band as filled polygon (TGraph)
    # Build polygon: go forward on upper, backward on lower
    y_up = mu + nsigma * std
    y_lo = mu - nsigma * std

    x_poly = np.concatenate([x_star, x_star[::-1]])
    y_poly = np.concatenate([y_up,   y_lo[::-1]])

    g_band = ROOT.TGraph(len(x_poly), x_poly.astype(np.float64), y_poly.astype(np.float64))
    g_band.SetName(f"g_band_{uid}")
    g_band.SetFillStyle(1001)
    g_band.SetFillColorAlpha(ROOT.kAzure - 9, 0.35)
    g_band.SetLineColorAlpha(ROOT.kAzure - 9, 0.0)
    c._objs.append(g_band)

    # Draw order
    g_data.Draw("AP")
    g_band.Draw("F SAME")  # filled polygon band
    g_mu.Draw("L SAME")
    g_data.Draw("P SAME")

    c.Modified()
    c.Update()

    # JSROOT inline render (same pattern as your working function)
    if hasattr(ROOT, "JSROOT") and hasattr(ROOT.JSROOT, "Draw"):
        return ROOT.JSROOT.Draw(c)

    return c

In [10]:
# Plot fit: dados + GP mean ± 2σ (ROOT)
# (Robusto: se você executar fora de ordem, este bloco cria o que faltar.)

if "x_star" not in globals():
    x_star = np.linspace(x.min(), x.max(), 400)

if ("mu_star" not in globals()) or ("var_star_latent" not in globals()):
    mu_star, var_star_latent = gp_predict_rbf(
        x_train=x, y_train=y, x_test=x_star,
        sigma_f=fit_real["sigma_f"],
        ell=fit_real["ell"],
        sigma_n=fit_real["sigma_n"],
    )

c_fit = plot_gp_fit_root_inline(
    x_train=x, y_train=y,
    x_star=x_star, mu=mu_star, var=var_star_latent,
    title="HEPData Figure 4a_cent7 — GP posterior mean ± 2σ (latent)",
    canvas_name="c_fit_real"
)
c_fit


## 6) Plot: comparação de incertezas σ_exp vs σ_GP,obs 

In [12]:
import ROOT
import uuid

def plot_uncertainty_compare_root_inline(
    x_points, sigma_exp_points,
    x_star, sigma_gp_obs_star,
    *,
    title="Uncertainty comparison: exp vs GP",
    canvas_name="c_unc",
    canvas_title="Uncertainty comparison",
    width=1100,
    height=450,
):
    ROOT.gROOT.SetBatch(True)
    ROOT.gStyle.SetOptStat(0)

    uid = uuid.uuid4().hex[:6]
    cname = f"{canvas_name}_{uid}"
    c = ROOT.TCanvas(cname, canvas_title, width, height)

    # Graph for experimental uncertainties (points)
    g_exp = ROOT.TGraph(len(x_points), np.asarray(x_points, dtype=np.float64), np.asarray(sigma_exp_points, dtype=np.float64))
    g_exp.SetMarkerStyle(20)
    g_exp.SetMarkerSize(1.0)
    g_exp.SetLineWidth(2)

    # Graph for GP uncertainties (curve)
    g_gp = ROOT.TGraph(len(x_star), np.asarray(x_star, dtype=np.float64), np.asarray(sigma_gp_obs_star, dtype=np.float64))
    g_gp.SetLineWidth(3)

    # Axes from a dummy frame
    xmin = float(min(np.min(x_points), np.min(x_star)))
    xmax = float(max(np.max(x_points), np.max(x_star)))
    ymin = 0.0
    ymax = float(max(np.max(sigma_exp_points), np.max(sigma_gp_obs_star)) * 1.15)

    frame = c.DrawFrame(xmin, ymin, xmax, ymax)
    frame.SetTitle(title)
    frame.GetXaxis().SetTitle("p_{T} [GeV]")
    frame.GetYaxis().SetTitle("Uncertainty (1#sigma)")

    g_gp.Draw("L SAME")
    g_exp.Draw("P SAME")

    leg = ROOT.TLegend(0.58, 0.72, 0.88, 0.88)
    leg.SetBorderSize(0)
    leg.AddEntry(g_exp, "#sigma_{exp} (stat #oplus sys)", "p")
    leg.AddEntry(g_gp, "#sigma_{GP,obs} (curve)", "l")
    leg.Draw()

    c._graphs = [g_exp, g_gp, leg, frame]
    c.Update()
    return c

c_unc = None

# Garante que x_star e sigma_star_obs existam, mesmo se rodar fora de ordem
if "x_star" not in globals():
    x_star = np.linspace(x.min(), x.max(), 400)

if "sigma_star_obs" not in globals():
    # tenta usar var_star_latent se existir, senão recomputa
    if "var_star_latent" in globals():
        sigma_star_obs = np.sqrt(var_star_latent + fit_real["sigma_n"]**2)
    else:
        mu_tmp, var_tmp = gp_predict_rbf(
            x_train=x, y_train=y, x_test=x_star,
            sigma_f=fit_real["sigma_f"],
            ell=fit_real["ell"],
            sigma_n=fit_real["sigma_n"],
        )
        sigma_star_obs = np.sqrt(var_tmp + fit_real["sigma_n"]**2)

c_unc = plot_uncertainty_compare_root_inline(
    x_points=df["pT"].to_numpy(),
    sigma_exp_points=df["sigma_exp"].to_numpy(),
    x_star=x_star,
    sigma_gp_obs_star=sigma_star_obs,
    title="Figure 4a_cent7 — #sigma_{exp}(p_{T}) vs #sigma_{GP,obs}(p_{T})"
)
c_unc


## 7) Tabela: experimental vs GP 

In [13]:
table = pd.DataFrame({
    "pT": df["pT"].to_numpy(),
    "v0_exp": df["v0"].to_numpy(),
    "sigma_exp": df["sigma_exp"].to_numpy(),
    "mu_gp": np.asarray(mu_x, dtype=float),
    "sigma_gp_obs": np.asarray(sigma_x_obs, dtype=float),
})

table["residual"] = table["v0_exp"] - table["mu_gp"]
table["pull_exp"] = table["residual"] / (table["sigma_exp"] + 1e-30)
table["sigma_ratio_gp_over_exp"] = table["sigma_gp_obs"] / (table["sigma_exp"] + 1e-30)

# arredondar para visualização
table_show = table.copy()
for col in table_show.columns:
    table_show[col] = table_show[col].map(lambda v: float(v))

table_show.head(12)


,pT,v0_exp,sigma_exp,mu_gp,sigma_gp_obs,residual,pull_exp,sigma_ratio_gp_over_exp
0,0.55231,-0.044431,0.001370,-0.044532,0.001613,0.000101,0.073655,1.177536
1,0.64791,-0.035065,0.001155,-0.034502,0.001504,-0.000563,-0.487407,1.301875
2,0.74831,-0.023929,0.000980,-0.023873,0.001450,-0.000056,-0.056635,1.479851
3,0.84819,-0.013116,0.000814,-0.013255,0.001434,0.000139,0.170904,1.762252
4,0.94830,-0.002188,0.000733,-0.002619,0.001436,0.000431,0.587414,1.958547
5,1.09310,0.012981,0.000813,0.012649,0.001448,0.000332,0.408598,1.779733
6,1.29330,0.033574,0.001142,0.033282,0.001456,0.000292,0.255692,1.275009
7,1.49340,0.052630,0.001667,0.053025,0.001451,-0.000395,-0.237062,0.870349
8,1.69350,0.070958,0.001975,0.071572,0.001443,-0.000614,-0.311008,0.730668
9,1.89360,0.088492,0.002599,0.088665,0.001438,-0.000173,-0.066742,0.553309


In [1]:
# Salvar tabela CSV
out_csv = Path("gp_realdata_exp_vs_gp_table.csv")
table.to_csv(out_csv, index=False)
print("Saved:", out_csv.resolve())

# Salvar os fit (hiperparâmetros)
hyper_csv = Path("gp_realdata_hyperparams.csv")
pd.DataFrame([fit_real]).to_csv(hyper_csv, index=False)
print("Saved:", hyper_csv.resolve())


NameError: name 'Path' is not defined

In [15]:
import ROOT
import numpy as np
import uuid

def plot_gp_fit_with_exp_errors_root_inline(
    x_points, y_points, yerr_points,
    x_star, mu_star, var_star_latent,
    sigma_n,
    *,
    title="GP fit: data (with exp errors) + GP mean ± 2σ (obs)",
    canvas_name="c_fit_err",
    width=1100,
    height=450,
):
    ROOT.gROOT.SetBatch(True)
    ROOT.gStyle.SetOptStat(0)

    uid = uuid.uuid4().hex[:6]
    cname = f"{canvas_name}_{uid}"
    c = ROOT.TCanvas(cname, title, width, height)

    x_points = np.asarray(x_points, dtype=np.float64)
    y_points = np.asarray(y_points, dtype=np.float64)
    yerr_points = np.asarray(yerr_points, dtype=np.float64)

    x_star = np.asarray(x_star, dtype=np.float64)
    mu_star = np.asarray(mu_star, dtype=np.float64)
    var_star_latent = np.asarray(var_star_latent, dtype=np.float64)

    # --- banda OBS: sqrt(var_latent + sigma_n^2)
    sigma_band = np.sqrt(np.clip(var_star_latent + sigma_n**2, 0.0, np.inf))

    # --- dados com barras experimentais
    ex0 = np.zeros_like(x_points)
    g_data = ROOT.TGraphErrors(len(x_points), x_points, y_points, ex0, yerr_points)
    g_data.SetMarkerStyle(20)
    g_data.SetMarkerSize(1.0)
    g_data.SetLineWidth(2)

    # --- curva média
    g_mu = ROOT.TGraph(len(x_star), x_star, mu_star)
    g_mu.SetLineWidth(3)

    # --- banda ±2σ como TGraphAsymmErrors (com preenchimento)
    exs = np.zeros_like(x_star)
    eys = 2.0 * sigma_band
    g_band = ROOT.TGraphAsymmErrors(len(x_star), x_star, mu_star, exs, exs, eys, eys)
    g_band.SetFillStyle(1001)  # sólido
    g_band.SetFillColorAlpha(ROOT.kAzure+1, 0.25)
    g_band.SetLineWidth(1)

    # --- frame
    xmin = float(min(x_points.min(), x_star.min()))
    xmax = float(max(x_points.max(), x_star.max()))
    ymin = float(min((y_points - yerr_points).min(), (mu_star - 2*sigma_band).min()))
    ymax = float(max((y_points + yerr_points).max(), (mu_star + 2*sigma_band).max()))
    pad = 0.08 * (ymax - ymin + 1e-12)
    ymin -= pad
    ymax += pad

    frame = c.DrawFrame(xmin, ymin, xmax, ymax)
    frame.SetTitle(title)
    frame.GetXaxis().SetTitle("p_{T} [GeV]")
    frame.GetYaxis().SetTitle("v_{0}(p_{T})")

    # desenha: banda atrás, depois média, depois dados
    g_band.Draw("3 SAME")
    g_mu.Draw("L SAME")
    g_data.Draw("P SAME")

    leg = ROOT.TLegend(0.55, 0.72, 0.88, 0.88)
    leg.SetBorderSize(0)
    leg.AddEntry(g_data, "Data (stat #oplus sys)", "p")
    leg.AddEntry(g_mu, "GP mean", "l")
    leg.AddEntry(g_band, "GP #pm 2#sigma_{obs}", "f")
    leg.Draw()

    c._keep = [g_data, g_mu, g_band, leg, frame]
    c.Update()
    return c

# --- usa as variáveis do notebook
c_fit_err = plot_gp_fit_with_exp_errors_root_inline(
    x_points=df["pT"].to_numpy(),
    y_points=df["v0"].to_numpy(),
    yerr_points=df["sigma_exp"].to_numpy(),
    x_star=x_star,
    mu_star=mu_star,
    var_star_latent=var_star_latent,
    sigma_n=fit_real["sigma_n"],
    title="HEPData Figure 4a_cent7 — Data (errors) + GP mean ± 2σ (obs)",
    canvas_name="c_fit_err"
)
c_fit_err